In [ ]:
from torch.nn import *
from sklearn.datasets import make_classification
import numpy as np
import torch

torch.set_default_tensor_type(torch.cuda.FloatTensor)


class BatchNorm(BatchNorm1d):
    def forward(s, x):
        return super().forward(x.view(-1, x.size(-1))).view(x.shape)


def unpack(a, n):
    if n < 9:
        return np.unpackbits(a.astype("B")[..., None], -1)[..., -n:]
    return np.unpackbits(a.astype("H")[..., None].view("B"), -1)[..., [*range(8), *range(24 - n, 16)]]


def dat(n, l, b=64):
    return torch.tensor(
        np.stack(
            [
                np.concatenate((a[0].astype("f"), unpack(a[1], n).astype("f")), 1)
                for a in range(b)
                for a in [make_classification(l, n, n_informative=n, n_redundant=0, n_classes=2 ** n, n_clusters_per_class=1)]
            ]
        )
    )


def fastdat(n, C, l=0, b=4):
    l = 2 ** int(np.log2((l or n) * C))
    i = np.random.randint(0, C, (b, l))
    return torch.randn(b, l, n).add_(torch.randn(b, C, n).mul_(16)[torch.arange(b)[:, None], torch.tensor(i, dtype=torch.long)]), torch.tensor(unpack(i, np.log2(C).__ceil__()), dtype=torch.float)


def fastdatold(n, b=64):
    return torch.randn(b, 2 ** n, n), torch.tensor(np.unpackbits(np.random.rand(b, 2 ** n).argsort(1).view("B")[:, ::8, None], 2)[..., -n:], dtype=torch.float)


def los(x, y):
    return functional.binary_cross_entropy_with_logits(x, y.broadcast_to(x.shape))


def acc(x, y):
    x = (x.signbit() != y).all(-1)
    return x.sum() / x.numel()


def last_eval(m, d, x, y):
    w, W = m[0].weight.T.split((d.size(2), x.size(2)), 0)
    return m[1:](d @ w + x @ W), y

In [ ]:
class Tnn(Sequential):
    def forward(s, x):
        for _ in range(int(np.log2(x.size(1)))):
            x = super().forward(x.view(-1, x.size(-1) * 2)) + x.view(-1, 2, x.size(-1)).mean(1)
        return x


class TruCell(Sequential):
    def forward(s, x):
        k = super().forward(x.view(len(x), -1)).sigmoid_()
        return (x[:, 0] * k).add_(x[:, 1] * (1 - k))


class Tru(TruCell):
    def forward(s, x):
        for _ in range(int(np.log2(x.size(1)))):
            x = super().forward(x.view(-1, 2, x.size(-1)))
        return x

In [ ]:
def model(k, c, r=.5, h=1):
    return 

In [ ]:
n = 8
c = 4
s = torch.cuda.amp.GradScaler()
m = n * 2 ** c
M = Sequential(
    Linear(n + c, m * 4, 0),
    BatchNorm(m * 4, track_running_stats=0),
    Hardswish(1),
    Linear(m * 4, m),
    Tru(Linear(m * 2, m * 4, 0), BatchNorm1d(m * 4, track_running_stats=0), Hardswish(1), Linear(m * 4, m)),
)
m = Sequential(Linear(m + n, m * 4, 0), BatchNorm(m * 4, track_running_stats=0), Hardswish(1), Linear(m * 4, c))
o = torch.optim.Adam((*M.parameters(), *m.parameters()))
O = torch.optim.lr_scheduler.OneCycleLR(o, 0.01, 200000)
# M, m, o, O = torch.load("nnTreeTrainEval.pk")
sum(i.numel() for i in o.param_groups[0]["params"])

In [ ]:
try:
    for i in range(200):
        for i in range(1000):
            o.zero_grad()
            with torch.cuda.amp.autocast():
                a = fastdat(n, 2 ** c, b=1024)
                a = last_eval(m, M(torch.cat(a, -1))[:, None], *a)
                i = los(*a)
            s.scale(i).backward()
            s.step(o)
            s.update()
            O.step()
        print(i, acc(*a), o.param_groups[0]["lr"])
except KeyboardInterrupt:
    torch.save((M, m, o, O), "nnTreeTrainEval.pk")
    print("saved")